In [1]:
import pandas as pd

data_path = r"C:\Projects\IR\data\articles_mapped.h5"

df = pd.read_hdf(
    data_path,
    key="data",
    start=0,
    stop=5
)

df

,web_url,pub_date,text,section_name,mapped_section
_id,,,,,
nyt://article/0ab7e750-5a63-5ce3-8274-b61e5930ac05,https://www.nytimes.com/2005/01/01/arts/design...,2005-01-01 05:00:00+00:00,Techniques That Might Smile Upon Mona Lisa Art...,Arts,"Culture, Arts & Lifestyle"
nyt://article/0ccaa5bc-2e61-5965-937c-0e1a35be669b,https://www.nytimes.com/2005/01/01/nyregion/cu...,2005-01-01 05:00:00+00:00,"Cut on Fish Tank, Man Bleeds to Death Antonio ...",New York,International News
nyt://article/0d1d5b9a-9826-51cf-9f36-ff4437ad368a,https://www.nytimes.com/2005/01/01/world/world...,2005-01-01 05:00:00+00:00,"As Officials Falter, the New Rich Roll in to H...",World,International News
nyt://article/1413b21a-d4db-5588-9caf-9f93593643c2,https://www.nytimes.com/2005/01/01/world/afric...,2005-01-01 05:00:00+00:00,Sudan and Southern Rebels Sign Pact to End Civ...,World,International News
nyt://article/1e015c8d-9f7f-5f1c-a96f-43609868c2ae,https://www.nytimes.com/2005/01/01/politics/ju...,2005-01-01 05:00:00+00:00,Justice Dept. Toughens Rule on Torture Justice...,U.S.,Politics & Government


In [2]:
# Load all articles from the HDF5 dataset
df = pd.read_hdf(data_path, key="data")

# Display the number of rows and columns
print("Dataset shape:", df.shape)

# Display the total number of articles
print("Total articles:", len(df))

Dataset shape: (1007004, 5)
Total articles: 1007004


In [3]:
# Extract the publication year from each article
df["year"] = df["pub_date"].dt.year

# Count articles published each year
yearly_counts = df["year"].value_counts().sort_index()

display(yearly_counts)

year
2005     79493
2006    113462
2007     74006
2008     55447
2009      7454
2010     28988
2011     40199
2012     76209
2013     58658
2014     61792
2015     54928
2016     46782
2017     40477
2018     38878
2019     37609
2020     39513
2021     37337
2022     34581
2023     32172
2024     35751
2025     13268
Name: count, dtype: int64

In [4]:
# Count articles for each year and news category
year_category_table = pd.crosstab(
    df["year"],
    df["mapped_section"]
)

display(year_category_table)

mapped_section,Business & Economy,"Culture, Arts & Lifestyle",Health & Wellbeing,International News,Lifestyle & Leisure,Other,Politics & Government,Sports & Athletics,"Tech, Science & Education"
year,,,,,,,,,
2005,15841,11800,1418,25981,4048,289,10949,7138,2029
2006,26969,16321,1868,32215,4999,854,14688,11946,3602
2007,18128,9577,996,15592,4071,1179,11612,9040,3811
2008,13245,6510,1054,9707,3332,706,10292,7721,2880
2009,1283,1609,270,1556,806,81,694,750,405
2010,5333,6060,828,6440,1661,260,3122,3783,1501
2011,6382,7285,924,7253,1894,1061,6553,5253,3594
2012,11431,12955,1409,14210,3246,2068,12924,10485,7481
2013,9539,10675,1176,12013,3096,1924,8697,6802,4736


In [6]:
import sys
from pathlib import Path

project_root = Path(r"C:\Projects\IR\CS533_HW1")
sys.path.insert(0, str(project_root))

In [7]:
from src.monthly_counts import monthly_article_counts

monthly_totals = monthly_article_counts(df, years=[2009, 2025])

display(monthly_totals)

,article_count
month,
2009-01-01,0
2009-02-01,0
2009-03-01,0
2009-04-01,0
2009-05-01,0
2009-06-01,0
2009-07-01,0
2009-08-01,0
2009-09-01,0


In [9]:
sports_monthly = monthly_article_counts(
    df,
    years=[2023, 2024, 2025],
    categories=["Sports & Athletics"]
)

display(sports_monthly)

mapped_section,Sports & Athletics
month,
2023-01-01,169
2023-02-01,144
2023-03-01,185
2023-04-01,188
2023-05-01,175
2023-06-01,162
2023-07-01,173
2023-08-01,127
2023-09-01,97


In [10]:
# Assign each article to its time period
df["period"] = df["year"].apply(
    lambda year: "P1" if 2005 <= year <= 2014
    else "P2" if 2015 <= year <= 2025
    else "Outside"
)

# Count articles in each category for both periods
period_category_table = pd.crosstab(
    df["mapped_section"],
    df["period"]
)

display(period_category_table)

period,P1,P2
mapped_section,,
Business & Economy,119367,52164
"Culture, Arts & Lifestyle",94516,87790
Health & Wellbeing,11069,19449
International News,137829,84113
Lifestyle & Leisure,31106,21071
Other,9966,8716
Politics & Government,89309,79553
Sports & Athletics,68483,29028
"Tech, Science & Education",34063,29412


In [11]:
# Find articles with fewer than 10 tokens
short_articles = df["text"].fillna("").str.count(r"\S+") < 10

# Count them
print("Articles with fewer than 10 tokens:", short_articles.sum())

Articles with fewer than 10 tokens: 28799


In [12]:
# Remove articles containing fewer than 10 tokens
clean_df = df.loc[~short_articles].copy()

print("Original articles:", len(df))
print("Removed articles:", short_articles.sum())
print("Remaining articles:", len(clean_df))

Original articles: 1007004
Removed articles: 28799
Remaining articles: 978205


In [13]:
# Identify duplicate article text
duplicate_text = clean_df["text"].duplicated(keep="first")

# Count duplicates
print("Duplicate articles:", duplicate_text.sum())

# Remove duplicates
clean_df = clean_df.loc[~duplicate_text].copy()

print("Remaining articles:", len(clean_df))

Duplicate articles: 6914
Remaining articles: 971291


In [14]:
# Identify duplicate article URLs
duplicate_urls = clean_df["web_url"].duplicated(keep="first")

print("Duplicate URLs:", duplicate_urls.sum())

# Remove duplicate URLs
clean_df = clean_df.loc[~duplicate_urls].copy()

print("Final cleaned articles:", len(clean_df))

Duplicate URLs: 571
Final cleaned articles: 970720


# Part 1.2: Data Cleaning

Before preparing the articles for the search system, I cleaned the dataset to remove very short records and repeated articles.

## Cleaning results

| Step | Articles removed | Articles remaining |
|---|---:|---:|
| Original dataset | — | 1,007,004 |
| Remove articles with fewer than 10 whitespace-separated tokens | 28,799 | 978,205 |
| Remove duplicate article text | 6,914 | 971,291 |
| Remove duplicate URLs | 571 | **970,720** |

## What I did and why

1. **Removed short articles.** I counted whitespace-separated tokens in each article's text and removed records with fewer than 10. These records contain too little text to be useful for the later analysis.
2. **Removed duplicate text.** Some records had exactly the same article text. I kept the first occurrence so identical text would not be counted multiple times.
3. **Removed duplicate URLs.** Finally, I checked for repeated `web_url` values among the remaining records and kept the first occurrence of each URL.

The order matters because each step changes which records remain for the next check. For example, the duplicate-text count was calculated **after** removing short articles, not on the original dataset. Changing the order could change the number attributed to each cleaning step and which duplicate record is kept.

After all three steps, **970,720 articles remained**, matching the expected cleaned total in the assignment. I kept the original DataFrame separate while working with the cleaned copy so I could check the intermediate counts.


In [15]:
import re

sentence = "The U.S. President’s NEW policy in 2025!"

# Convert to lowercase
text = sentence.lower()

# Normalize the apostrophe
text = text.replace("’", "'")

# Remove periods between letters in acronyms
text = re.sub(
    r"\b(?:[a-z]\.){2,}",
    lambda m: m.group().replace(".", ""),
    text
)

# Remove possessive endings
text = re.sub(r"'s\b", "", text)

# Extract words and numbers
tokens = re.findall(r"[a-z]+|\d+", text)

print(tokens)

['the', 'us', 'president', 'new', 'policy', 'in', '2025']


In [18]:
# Select two articles from our cleaned dataset
examples = clean_df["text"].iloc[:2]

for i, article in enumerate(examples, start=1):
    print(f"\nARTICLE {i}")
    print("Original:", article[:300])
    print("Processed:", re.findall(
        r"[a-z]+|\d+",
        re.sub(
            r"'s\b", "",
            re.sub(
                r"\b(?:[a-z]\.){2,}",
                lambda m: m.group().replace(".", ""),
                article.lower().replace("’", "'")
            )
        )
    )[:50])


ARTICLE 1
Original: Techniques That Might Smile Upon Mona Lisa Article on research conducted by Smithsonian Institution engineer and painting conservator Marion Mecklenburg, who is seeking better ways to preserve priceless art works, including those like Mona Lisa that are painted on thin wood panels; Mecklenburg and a
Processed: ['techniques', 'that', 'might', 'smile', 'upon', 'mona', 'lisa', 'article', 'on', 'research', 'conducted', 'by', 'smithsonian', 'institution', 'engineer', 'and', 'painting', 'conservator', 'marion', 'mecklenburg', 'who', 'is', 'seeking', 'better', 'ways', 'to', 'preserve', 'priceless', 'art', 'works', 'including', 'those', 'like', 'mona', 'lisa', 'that', 'are', 'painted', 'on', 'thin', 'wood', 'panels', 'mecklenburg', 'and', 'assistant', 'have', 'found', 'that', 'maintaining', 'strict']

ARTICLE 2
Original: Cut on Fish Tank, Man Bleeds to Death Antonio Perez bleeds to death in his Manhattan apartment after he knocks over fish tank and cuts his arm on broken g

In [21]:
from src.preprocessing import preprocess_text

preprocess_text("The N.B.A. team's 2025 pre-season.")
# Expected Result: ['the', 'nba', 'team', '2025', 'pre', 'season']



['the', 'nba', 'team', '2025', 'pre', 'season']

In [24]:
import re

# Find an article containing a dotted abbreviation
pattern = r"\b(?:U\.S\.|N\.B\.A\.|F\.B\.I\.)"

matches = clean_df["text"].str.contains(
    pattern, regex=True, na=False
)

article = clean_df.loc[matches, "text"].iloc[0]

# Extract the text surrounding the abbreviation
match = re.search(pattern, article)

excerpt = article[
    max(0, match.start() - 70):match.end() + 100
]

print("ORIGINAL:")
print(excerpt)

print("\nPROCESSED:")
print(preprocess_text(excerpt))

ORIGINAL:
U.S. Vows Big Increase in Aid for Victims of Asian Disaster Pres Bush will increase emergency aid to str

PROCESSED:
['us', 'vows', 'big', 'increase', 'in', 'aid', 'for', 'victims', 'of', 'asian', 'disaster', 'pres', 'bush', 'will', 'increase', 'emergency', 'aid', 'to', 'str']
